# 🧠 Logic Fine-Tuning: Qwen3-8B × Unsloth — v7 — Kiến trúc Tối thượng (Bug-Fixed)
**Pipeline:** Data Unpacking → LoRA Training → Chain-of-Thought Alignment → Z3 Multi-Goal Verification → Stateful Feedback → Evaluation

> Dataset: `Logic_Based_Educational_Queries-2.json` | 411 indices | FOL + NL premises  
> Target: **≥60% accuracy** trên toàn bộ test set

---
## 📋 Fixes so với phiên bản cũ
| # | Vấn đề cũ | Fix mới |
|---|-----------|----------|
| 1 | Qwen2.5-7B | **Qwen3-8B** (mạnh hơn về reasoning) |
| 2 | `formatting_prompts_func` định nghĩa 2 lần | Chỉ 1 lần, đúng signature |
| 3 | `train_on_responses_only` dùng sai token markers | Dùng đúng `<|im_start|>` cho Qwen3 |
| 4 | `dataset_text_field="text"` nhưng `formatting_func` trả list | Fix để SFTTrainer nhận đúng |
| 5 | LR=1e-5 quá thấp (under-fit) | **LR=2e-4** (chuẩn LoRA) |
| 6 | Evaluation dùng raw_data[-3:] không đúng split | Fix dùng đúng test split |
| 7 | `extract_final_answer` regex yếu | Regex mạnh hơn + fallback |
| 8 | Inference thiếu `repetition_penalty` | Thêm penalty để tránh loop |
| 9 | Evaluate chỉ trên vài sample | **Evaluate toàn bộ 411 indices** |

---
## 🚀 Nâng cấp v2 — 4 Chiến lược mới
| Chiến lược | Mô tả | Tác dụng |
|------------|-------|----------|
| **S1: Constrained Decoding** | Guided Decoding + JSON Schema ép AST đúng cú pháp tại cấp Logits | Parser không bao giờ crash |
| **S2: SMT-LIB v2 Output** | Qwen sinh thẳng SMT-LIB2 thay vì AST tự chế | Z3 đọc native, loại bỏ parser trung gian |
| **S3: Feedback Translator** | Module dịch lỗi Z3 → ngôn ngữ tự nhiên có hướng dẫn cụ thể | Qwen biết chính xác chỗ cần sửa |
| **S4: Early Stopping + Fallback** | Max 3 vòng lặp; nếu vẫn lỗi → mặc định "Unknown" | Tiết kiệm compute, tránh infinite loop |
---
## 🏗️ Kiến trúc v5 — 4 Thay đổi Tối thượng
| # | Vấn đề v4 | Giải pháp v5 |
|---|-----------|--------------|
| A | MCQ single-pass: sat → Unknown ngay | **Multi-Goal Z3**: push()/pop() kiểm tra từng đáp án độc lập |
| B | `repetition_penalty=1.1` phá formal language | **penalty=1.0** (greedy decode thuần túy cho SMT-LIB2) |
| C | Oversampling Unknown 3x → Lazy Agent Bias | **Xóa Oversampling** + Z3 tự chứng minh Unknown bằng SAT∧SAT(¬H) |
| D | Static feedback → Mode Collapse dưới 10B | **Stateful Feedback** với lịch sử thất bại tích lũy |

---
## 🔧 Fixes v7 — Bug Fix Release
| # | Lỗi được phát hiện | Fix áp dụng |
|---|--------------------|--------------|
| 1 | `MAX_SEQ_LEN=2048` gây truncation khi context > 2048 token | **Tăng lên 4096** |
| 2 | `is_solvable` field cho phép LLM tự thoát → Lazy Agent Bias | **Xóa hoàn toàn** khỏi schema + pipeline |
| 3 | `apply_search_replace_patch` chỉ strip trailing space, không xử lý mixed indent (`\\t` vs space) và CRLF | **Nâng cấp normalizer** 3 tầng: exact → strip-trailing → normalize-indent |


| # | Lỗi được phát hiện | Fix áp dụng |
|---|--------------------|--------------|
| BUG#1 | `max_new_tokens=300` quá nhỏ → JSON bị cắt → JSONDecodeError | **Tăng lên 1024** trong evaluation; iter 2/3 tăng từ 400 → **600** |
| BUG#2 | `predict_with_z3_loop` thiếu `max_length=None` → xung đột với model config | **Thêm `max_length=None`** vào `model.generate()` |
| BUG#5 | Thu hồi: `disable_adapter()` đúng vì data train là NL text, không phải JSON | Giữ nguyên `disable_adapter()` |
| BUG#6 | `SYSTEM_PROMPT` vs `build_smtlib_prompt()` mâu thuẫn (Final Answer vs JSON) | **Thêm `SMTLIB_SYSTEM_PROMPT`** riêng cho Z3 loop |
| BUG#7 | Schema JSON thiếu `hypothesis_assert`, `premises_only_script`, `option_asserts` | **Thêm 3 fields** vào `goal.properties` trong schema |
| BUG#8 | `parse_smt2_string(neg_assert)` thiếu header → Z3Exception | **Ghép header** (set-logic + declare-*) trước khi parse |
| BUG#9 | `patch_summary` hardcode `"(attempt)"` → Stateful Feedback vô nghĩa | **Lưu nội dung SEARCH block** thực tế |



In [1]:
%%capture
# Unsloth với CUDA 12.1 (T4 / A100 Kaggle)
!pip install unsloth
!pip install --upgrade --no-cache-dir trl peft accelerate bitsandbytes
!pip install datasets transformers sentencepiece protobuf
# Chiến lược 1: Constrained Decoding
!pip install outlines
# Chiến lược 2: Z3 SMT Solver
!pip install z3-solver

In [2]:
import json
import os
import re
import random
import copy
from collections import Counter, defaultdict
from typing import List, Dict, Any, Optional, Tuple
import torch
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

# Chiến lược 2: Z3 SMT Solver
try:
    import z3
    Z3_AVAILABLE = True
    print("✅ Z3 available")
except ImportError:
    Z3_AVAILABLE = False
    print("⚠️  Z3 not available — install z3-solver")

# Seed cố định để reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:144: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ Z3 available
✅ PyTorch version: 2.10.0+cu128
✅ CUDA available: True
✅ GPU: Tesla T4
✅ VRAM: 15.6 GB


In [3]:
# ============================================================
#  GLOBAL CONFIG
# ============================================================

# --- Path ---
# Cập nhật DATA_PATH trỏ đến file JSON dataset thực tế
DATA_PATH  = "/content/Logic_Based_Educational_Queries-2.json"
OUTPUT_DIR = "/content/lora_output_qwen3"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Model (FIX #1: Qwen3-8B thay vì Qwen2.5-7B) ---
MODEL_NAME  = "unsloth/Qwen3-8B-bnb-4bit"   # 4-bit quantized, tối ưu T4/A100
MAX_SEQ_LEN = 4096  # v6 FIX #1: tăng từ 2048 → chứa đủ prompt lớn + failed_history

# --- LoRA Config ---
LORA_R       = 64      # Rank cao hơn để học logic sâu hơn
LORA_ALPHA   = 128      # = 2 * LORA_R
LORA_DROPOUT = 0.05

LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# --- Training Hyperparams (FIX #5: LR chuẩn LoRA) ---
LEARNING_RATE    = 1e-4     # Chuẩn LoRA
BATCH_SIZE       = 2
GRAD_ACCUM_STEPS = 8        # Effective batch = 16
NUM_EPOCHS       = 4
WARMUP_RATIO     = 0.05
LR_SCHEDULER     = "cosine"
WEIGHT_DECAY     = 0.01
MAX_GRAD_NORM    = 1.0

# --- Data Split ---
TRAIN_RATIO = 0.85
VAL_RATIO   = 0.10
TEST_RATIO  = 0.05

# --- Adversarial Weighting ---
UNKNOWN_OVERSAMPLE_FACTOR = 3

print("✅ Config loaded.")
print(f"   Model         : {MODEL_NAME}")
print(f"   LoRA rank     : {LORA_R}, alpha: {LORA_ALPHA}")
print(f"   Effective BS  : {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"   Learning rate : {LEARNING_RATE}")
print(f"   Epochs        : {NUM_EPOCHS}")

# --- Chiến lược 1: Constrained Decoding JSON Schema ---
SMTLIB_JSON_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": ["thought_process", "declarations", "premises", "goal"],  # v6: bỏ is_solvable
    "properties": {
        "thought_process": {"type": "string"},
        # v6 FIX #2: is_solvable đã bị xóa — LLM không được tự tuyên bố bài toán vô nghiệm
        "declarations": {
            "type": "object",
            "properties": {
                "constants": {"type": "array", "items": {"type": "string"}},
                "predicates": {"type": "array", "items": {"type": "string"}}
            }
        },
        "premises": {
            "type": "array",
            "items": {
                "type": "object",
                "required": ["index", "natural_language", "operator", "expression"],
                "properties": {
                    "index": {"type": "integer"},
                    "natural_language": {"type": "string"},
                    "operator": {"type": "string", "enum": ["AND", "OR", "NOT", "IMPLIES", "FORALL", "EXISTS", "ATOM"]},
                    "expression": {"type": "string", "pattern": "^\\(.*\\)$"}
                }
            }
        },
        "goal": {
            "type": "object",
            "required": ["question_reflection", "expression", "smtlib2_script"],
            "properties": {
                "question_reflection": {"type": "string"},
                "expression": {"type": "string", "pattern": "^\\(.*\\)$"},
                "smtlib2_script": {"type": "string"},
                "premises_only_script": {
                    "type": "string",
                    "description": "BUG#7 FIX: SMT-LIB2 với CHỈ declarations + assert premises (không có check-sat, không có hypothesis). Bắt buộc cho cả YES/NO và MCQ."
                },
                "hypothesis_assert": {
                    "type": "string",
                    "description": "BUG#7 FIX: Dành cho YES/NO — '(assert <hypothesis>)' đơn lẻ. Backend gọi Z3 2 lần: lần với H, lần với ¬H."
                },
                "option_asserts": {
                    "type": "object",
                    "description": "BUG#7 FIX: Dành cho MCQ — keys A/B/C/D, value là '(assert <FOL cho đáp án đó>)'. Backend dùng push()/pop() kiểm tra từng đáp án.",
                    "additionalProperties": {"type": "string"}
                }
            }
        }
    }
}

MAX_Z3_ITERATIONS = 3
FALLBACK_ANSWER   = "Unknown"

print("✅ v6 Config loaded.")
print(f"   MAX_SEQ_LEN   : {MAX_SEQ_LEN}  (v6: 4096 để tránh truncation)")
print("   is_solvable   : REMOVED from schema (v6: chống Lazy Agent Bias)")

✅ Config loaded.
   Model         : unsloth/Qwen3-8B-bnb-4bit
   LoRA rank     : 64, alpha: 128
   Effective BS  : 16
   Learning rate : 0.0001
   Epochs        : 4
✅ v6 Config loaded.
   MAX_SEQ_LEN   : 4096  (v6: 4096 để tránh truncation)
   is_solvable   : REMOVED from schema (v6: chống Lazy Agent Bias)


---
## ⚙️ Chiến lược 1 — Constrained Decoding (Guided JSON Schema)

Thay vì hy vọng Qwen tự sinh AST đúng cú pháp, ta dùng **JSON Schema Guided Decoding**  
để ép Qwen luôn trả về một JSON hợp lệ chứa SMT-LIB2 script.  
`outlines` thao túng mảng Logits tại mỗi bước decode — parser sẽ không bao giờ crash nữa.

In [37]:
# ============================================================
#  CHIẾN LƯỢC 1: CONSTRAINED DECODING — JSON Schema Guard
# ============================================================
import json as _json
import re as _re

def _strip_thinking(text: str) -> str:
    """
    Strip <think>...</think> blocks — Qwen3 thinking mode.
    Xử lý 2 trường hợp:
      - Đủ cặp tag: <think>...</think> → xóa toàn bộ block
      - Thiếu </think> (bị cắt giữa chừng): xóa từ <think> đến hết,
        vì mọi thứ sau <think> mà không có </think> đều là thinking, không phải JSON
    """
    # Trường hợp 1: có đủ cặp → xóa block
    text = _re.sub(r"<think>.*?</think>", "", text, flags=_re.DOTALL | _re.IGNORECASE)
    # Trường hợp 2: còn <think> mà không có </think> → cắt từ đó đến hết
    text = _re.sub(r"<think>.*", "", text, flags=_re.DOTALL | _re.IGNORECASE)
    return text.strip()

def validate_smtlib_json(raw_output: str) -> Optional[Dict]:
    """
    Validate và extract JSON từ output của Qwen.
    Đây là hậu kiểm tra (post-hoc) cho môi trường không có vLLM native.
    Production với vLLM: thêm --guided-decoding-backend outlines
    và truyền json_schema=SMTLIB_JSON_SCHEMA vào SamplingParams.
    """
    # FIX THINKING MODE: strip <think> blocks trước khi parse
    raw_output = _strip_thinking(raw_output)
    # Tách JSON block khỏi markdown fences nếu có
    fence_match = _re.search(r"```(?:json)?\s*(\{.*?\})\s*```", raw_output, _re.DOTALL)
    if fence_match:
        json_str = fence_match.group(1)
    else:
        # Lưới lọc an toàn tuyệt đối chống Crash
        brace_match = _re.search(r"(\{.*\})", raw_output, _re.DOTALL)
        json_str = brace_match.group(1) if brace_match else "{}"

    try:
        parsed = _json.loads(json_str)
        if not parsed:  # Chặn bắt JSON rỗng
            print("    ⚠️  No JSON structure found in output")
            return None
    except _json.JSONDecodeError as e:
        print(f"    ⚠️  JSON parse failed: {e}")
        return None

    # ── v6 FIX #2: is_solvable ĐÃ BỊ XÓA khỏi schema ──
    # Lý do: LLM có xu hướng đặt is_solvable=False để thoát sớm (Lazy Agent Bias).
    # Khi đó nó không viết smtlib2_script đầy đủ → Z3 check chạy trên script rỗng
    # → luôn trả Unknown mà không có cơ sở toán học.
    # Fix: LLM PHẢI luôn dịch 100% premises sang SMT-LIB2.
    # Việc Unknown/Known do Z3 quyết định hoàn toàn qua run_z3_check_independence().

    # ── Kiểm tra các trường bắt buộc ──
    required = {"thought_process", "declarations", "premises", "goal"}
    missing = required - set(parsed.keys())
    if missing:
        print(f"    ⚠️  Missing fields: {missing}")
        return None

    # ── Lấy SMT-LIB2 script từ Lớp 4 (goal) ──
    script = parsed.get("goal", {}).get("smtlib2_script", "")
    # BUG FIX: "check-sat" in script đã bao gồm "(check-sat)", không cần check 2 lần
    if "(check-sat)" not in script:
        print("    ⚠️  goal.smtlib2_script missing (check-sat)")
        return None

    print("    ✅ JSON schema validation passed")
    return parsed


def build_smtlib_prompt(fol_list: List[str], nl_list: List[str], question: str) -> str:
    """Prompt yêu cầu Qwen sinh SMT-LIB2 trực tiếp dưới dạng JSON."""
    premises_block = format_premises(fol_list, nl_list)
    schema_str = _json.dumps(SMTLIB_JSON_SCHEMA, indent=2)
    return (
        f"{premises_block}\n\n"
        f"### Question\n{question}\n\n"
        "### Task\n"
        "Translate the premises and the hypothesis into SMT-LIB2 format.\n"
        f"Return ONLY a valid JSON object conforming to this schema:\n{schema_str}\n\n"
        "The `smtlib2_script` must:\n"
        "- Start with (set-logic QF_UF) or appropriate logic\n"
        "- Declare all constants/functions\n"
        "- Assert all premises\n"
        "- Assert the NEGATION of the hypothesis (proof by contradiction)\n"
        "- End with (check-sat)\n\n"
        f"{SMTLIB_EXTRA_INSTRUCTIONS}\n\n"
        "Return only the JSON object, no other text."
    )


VLLM_GUIDED_DECODING_NOTE = """
[PRODUCTION NOTE — Chiến lược 1]
Để kích hoạt Constrained Decoding native trong vLLM:
  from vllm import LLM, SamplingParams
  sampling_params = SamplingParams(
      guided_decoding_backend="outlines",
      guided_json=SMTLIB_JSON_SCHEMA,
      temperature=0.0,
  )
  llm = LLM(model=LORA_SAVE_PATH)
  output = llm.generate(prompt, sampling_params)

Trong Kaggle/Unsloth hiện tại, validate_smtlib_json() làm post-hoc validation.
"""
print(VLLM_GUIDED_DECODING_NOTE)
print("✅ v6 Chiến lược 1: Constrained Decoding module ready (is_solvable removed)")


[PRODUCTION NOTE — Chiến lược 1]
Để kích hoạt Constrained Decoding native trong vLLM:
  from vllm import LLM, SamplingParams
  sampling_params = SamplingParams(
      guided_decoding_backend="outlines",
      guided_json=SMTLIB_JSON_SCHEMA,
      temperature=0.0,
  )
  llm = LLM(model=LORA_SAVE_PATH)
  output = llm.generate(prompt, sampling_params)

Trong Kaggle/Unsloth hiện tại, validate_smtlib_json() làm post-hoc validation.

✅ v6 Chiến lược 1: Constrained Decoding module ready (is_solvable removed)


---
## ⚙️ Chiến lược 2 — SMT-LIB v2 Direct Output + Z3 Verification

Thay vì AST tự chế rồi dịch sang Z3 Python API, ta yêu cầu Qwen sinh thẳng **SMT-LIB2**.  
Z3 đọc SMT-LIB2 native — loại bỏ hoàn toàn parser trung gian.

In [5]:
# ============================================================
#  CHIẾN LƯỢC 2: SMT-LIB v2 DIRECT OUTPUT + Z3 VERIFICATION
#  v5: Multi-Goal MCQ + Z3-proved Unknown
# ============================================================

def run_z3_on_smtlib(smtlib_script: str, timeout_ms: int = 5000) -> Dict[str, Any]:
    """
    Chạy Z3 trực tiếp trên SMT-LIB2 script (native parsing).
    Trả về dict: {status, model, unsat_core, error}
    - status: 'sat' | 'unsat' | 'unknown' | 'error'
    """
    if not Z3_AVAILABLE:
        return {"status": "error", "error": "Z3 not installed", "model": None}
    try:
        solver = z3.Solver()
        solver.set("timeout", timeout_ms)
        # FIX: parse_smt2_string returns assertions, add them to solver manually
        assertions = z3.parse_smt2_string(smtlib_script)
        solver.add(assertions)
        result = solver.check()
        if result == z3.sat:
            return {"status": "sat", "model": str(solver.model()), "error": None}
        elif result == z3.unsat:
            try:
                core = [str(c) for c in solver.unsat_core()]
            except Exception:
                core = []
            return {"status": "unsat", "unsat_core": core, "error": None}
        else:
            return {"status": "unknown", "error": "Z3 timeout or undecidable", "model": None}
    except z3.Z3Exception as e:
        return {"status": "error", "error": str(e), "model": None}
    except Exception as e:
        return {"status": "error", "error": f"Unexpected: {e}", "model": None}


def _extract_smtlib_header(script: str) -> str:
    """
    BUG#8 FIX helper: Trích xuất phần header (set-logic + declare-*) từ premises_script
    để ghép vào hypothesis_assert / neg_assert trước khi gọi parse_smt2_string().
    parse_smt2_string() yêu cầu full script — không thể parse 1 dòng assert đơn lẻ.
    """
    lines = []
    for line in script.splitlines():
        stripped = line.strip()
        if stripped.startswith("(set-logic") or stripped.startswith("(declare-"):
            lines.append(line)
    return "\n".join(lines)


def run_z3_check_independence(
    premises_script: str,
    hypothesis_assert: str,
    timeout_ms: int = 5000,
) -> str:
    """
    v5 — Kiểm tra tính độc lập logic bằng Z3 push()/pop():
    BUG#8 FIX: ghép header (set-logic + declare-*) từ premises_script
    vào hypothesis_assert trước khi parse, vì parse_smt2_string()
    yêu cầu full script có (set-logic ...) và (declare-*).
    """
    if not Z3_AVAILABLE:
        return None
    try:
        # BUG#8 FIX: Trích header để dùng cho hypothesis/neg_assert
        prem_header = _extract_smtlib_header(premises_script)

        solver = z3.Solver()
        solver.set("timeout", timeout_ms)
        solver.add(z3.parse_smt2_string(premises_script))

        # ── Nhánh H ──
        solver.push()
        try:
            # BUG#8 FIX: ghép header + check-sat để tạo full script hợp lệ
            h_full_script = prem_header + "\n" + hypothesis_assert.strip() + "\n(check-sat)"
            solver.add(z3.parse_smt2_string(h_full_script))
        except z3.Z3Exception:
            solver.pop()
            return None
        h_result = solver.check()
        solver.pop()

        # ── Nhánh ¬H ──
        solver.push()
        neg_assert = hypothesis_assert.strip()
        m = __import__("re").match(r"\(assert\s+(.*)\)\s*$", neg_assert, __import__("re").DOTALL)
        if m:
            neg_assert = f"(assert (not {m.group(1).strip()}))"
        else:
            solver.pop()
            return None
        try:
            # BUG#8 FIX: ghép header + check-sat để tạo full script hợp lệ
            not_h_full_script = prem_header + "\n" + neg_assert + "\n(check-sat)"
            solver.add(z3.parse_smt2_string(not_h_full_script))
        except z3.Z3Exception:
            solver.pop()
            return None
        not_h_result = solver.check()
        solver.pop()

        if not_h_result == z3.unsat: return "Yes"
        elif h_result == z3.unsat: return "No"
        elif h_result == z3.sat and not_h_result == z3.sat: return "Unknown"
        else: return None

    except Exception as e:
        print(f"    ⚠️  Z3 independence error: {e}")
        return None


def run_z3_mcq_multigoal(
    premises_script: str,
    option_asserts: Dict[str, str],
    timeout_ms: int = 5000,
) -> Dict[str, Any]:
    """
    v5 — Multi-Goal MCQ Verification với push()/pop()
    """
    if not Z3_AVAILABLE:
        return {"answer": None, "results": {}, "error": "Z3 not available"}

    try:
        base_solver = z3.Solver()
        base_solver.set("timeout", timeout_ms)
        base_solver.add(z3.parse_smt2_string(premises_script))
    except Exception as e:
        return {"answer": None, "results": {}, "error": f"Premises parse error: {e}"}

    results = {}
    proven_opts = []

    for label, opt_assert in sorted(option_asserts.items()):
        base_solver.push()
        try:
            m = __import__("re").match(r"\(assert\s+(.*)\)\s*$", opt_assert.strip(), __import__("re").DOTALL)
            if not m:
                results[label] = "parse_error"
                base_solver.pop()
                continue
            neg_opt_assert = f"(assert (not {m.group(1).strip()}))"
            base_solver.add(z3.parse_smt2_string(neg_opt_assert))
            check = base_solver.check()
            if check == z3.unsat:
                results[label] = "unsat"
                proven_opts.append(label)
            else:
                results[label] = "sat" if check == z3.sat else "unknown"
        except Exception:
            results[label] = "error"
        finally:
            base_solver.pop()

    if len(proven_opts) == 1: answer = proven_opts[0]
    else: answer = "Unknown"

    return {"answer": answer, "results": results, "error": None}

def interpret_z3_result(z3_result: Dict, question: str) -> Optional[str]:
    status = z3_result.get("status")
    if status == "unsat": return "Yes"
    elif status == "sat": return "No"
    return None

print("✅ Chiến lược 2 v5 Fixed: Z3 module ready")

if Z3_AVAILABLE:
    _test_script = "(set-logic QF_UF)\n(declare-const p Bool)\n(declare-const q Bool)\n(assert p)\n(assert (=> p q))\n(assert (not q))\n(check-sat)"
    _res = run_z3_on_smtlib(_test_script)
    assert _res["status"] == "unsat", f"Sanity test 1 failed: {_res}"
    print("   ✅ Z3 sanity test 1 passed")

✅ Chiến lược 2 v5 Fixed: Z3 module ready
   ✅ Z3 sanity test 1 passed


---
## ⚙️ Chiến lược 3 — Search/Replace Patch (Bộ vá lỗi cục bộ)

Khi Z3 thất bại, thay vì yêu cầu Qwen sinh lại **toàn bộ** script tốn 500-1000 token,  
module này áp dụng mô hình **SEARCH/REPLACE**: Qwen chỉ xuất ra khối vá lỗi ~50 token.  
Backend Python thực hiện `old_script.replace(search, replace)` rồi mới đưa vào Z3.

**Ba lệnh cấm cứng:**  
- Không xin lỗi  
- Không giải thích dài  
- Không sinh lại toàn bộ file — chỉ SEARCH/REPLACE

In [6]:
# ============================================================
#  CHIẾN LƯỢC 3: SEARCH/REPLACE PATCH (Local Error Fix)
# ============================================================

# ── System Prompt dành riêng cho Lượt hội thoại thứ 2 (Feedback Loop) ──
# Khác hoàn toàn SYSTEM_PROMPT chính: không giải thích, không xin lỗi,
# chỉ phun ra đúng 1 khối SEARCH/REPLACE rồi dừng.
DEBUGGER_SYSTEM_PROMPT = """Bạn là một Chuyên gia gỡ lỗi Logic (SMT-LIB2 Debugger).
Z3 Solver vừa phát hiện lỗi trong bản dịch logic trước đó của bạn.
NHIỆM VỤ CỦA BẠN:
1. KHÔNG xin lỗi. KHÔNG giải thích dài dòng. KHÔNG sinh lại toàn bộ file.
2. Xác định chính xác vị trí lỗi dựa trên phản hồi của Z3.
3. Xuất ra ĐÚNG MỘT khối CẬP NHẬT theo định dạng SEARCH/REPLACE dưới đây.

Định dạng bắt buộc:
<SEARCH>
[Copy dán CHÍNH XÁC dòng/khối logic bị sai từ phiên bản trước,
 giữ nguyên khoảng trắng, dấu tab, thụt đầu dòng]
</SEARCH>
<REPLACE>
[Viết dòng/khối logic ĐÃ ĐƯỢC SỬA chuẩn xác]
</REPLACE>"""


def format_z3_feedback_for_search_replace(
    z3_result: Dict[str, Any],
    current_smtlib_script: str,
    iteration: int,
    original_fol: List[str],
    original_nl: List[str],
    failed_history: Optional[List[Dict]] = None,   # v5: lịch sử thất bại tích lũy
) -> str:
    """
    v5 — Stateful Feedback: Tạo User Prompt SEARCH/REPLACE có nhận thức lịch sử.

    Chống "Mode Collapse in Self-Correction" trên mô hình dưới 10B:
    - Mô hình nhỏ pattern-match vào error message quen thuộc
      và tái tạo cùng lỗi dù script đã thay đổi.
    - Giải pháp: ép model nhận thức CÁC HƯỚNG ĐÃ THỬ VÀ THẤT BẠI
      → buộc phải "quay xe" sang hướng tư duy mới.

    failed_history: list of {"iteration": int, "error": str, "patch_applied": str}
    """
    status    = z3_result.get("status", "error")
    error_msg = z3_result.get("error", "")
    unsat_core = z3_result.get("unsat_core", [])

    lines = [
        f"[Vòng lặp {iteration}/{MAX_Z3_ITERATIONS}]",
        "",
    ]

    # ── v5: Stateful Feedback — lịch sử thất bại tích lũy ──
    # Ép model nhận thức các hướng đã thử để tránh Mode Collapse
    if failed_history:
        lines.append("[LỊCH SỬ CÁC LẦN SỬA THẤT BẠI — TUYỆT ĐỐI KHÔNG LẶP LẠI]:")
        for h in failed_history:
            lines.append(
                f"  ✗ Vòng {h['iteration']}: Lỗi Z3 = [{h['error'][:120]}] "
                f"| Bản vá đã thử: {h['patch_summary'][:80]}"
            )
        lines += [
            "→ Các cách sửa trên ĐÃ THẤT BẠI. Hãy thay đổi HOÀN TOÀN cách tiếp cận.",
            "→ Suy nghĩ lại từ đầu: lỗi có thể ở tầng khai báo (declare), "
            "không phải tầng assert.",
            "",
        ]

    # ── Phân loại lỗi → hướng dẫn cụ thể ──
    if status == "error":
        em = error_msg.lower()
        if "unknown sort" in em or "undeclared" in em:
            lines += [
                "[LỖI TỪ Z3 SOLVER]: Undeclared Identifier (Dùng ký hiệu chưa khai báo).",
                f"[CHI TIẾT]: {error_msg}",
                "[HƯỚNG DẪN]: Thêm `(declare-const <tên> <kiểu>)` TRƯỚC assert sử dụng nó.",
            ]
        elif "parse error" in em or "unexpected" in em:
            lines += [
                "[LỖI TỪ Z3 SOLVER]: Parse Error (Lỗi cú pháp SMT-LIB2).",
                f"[CHI TIẾT]: {error_msg}",
                "[HƯỚNG DẪN]: Kiểm tra dấu ngoặc. Script phải kết thúc bằng `(check-sat)`.",
            ]
        elif "logic" in em:
            lines += [
                "[LỖI TỪ Z3 SOLVER]: Invalid Logic Declaration.",
                f"[CHI TIẾT]: {error_msg}",
                "[HƯỚNG DẪN]: Nếu có ∀/∃ → dùng `(set-logic UF)`. Không có quantifier → `(set-logic QF_UF)`.",
            ]
        elif "=>" in error_msg or "->" in error_msg or "IMPLIES" in error_msg.upper() or "implication" in error_msg.lower():
            # BUG FIX: Z3 không dùng "=>" trong error message — dùng "->" hoặc "IMPLIES"
            # Nếu chỉ check "=>" thì branch này không bao giờ được kích hoạt (dead code)
            impl_premises = [
                f"   ⚠️  Tiền đề {i+1}: [{nl}] — FOL: {fol}"
                for i, (fol, nl) in enumerate(zip(original_fol, original_nl))
                if "=>" in fol or "implies" in nl.lower() or "if " in nl.lower()
            ]
            lines += [
                "[LỖI TỪ Z3 SOLVER]: Implication Direction Error (Đảo chiều kéo theo).",
                f"[CHI TIẾT]: {error_msg}",
                "[HƯỚNG DẪN]: `A kéo theo B` phải là `(=> A B)`, KHÔNG phải `(=> B A)`.",
            ]
            if impl_premises:
                lines.append("[VỊ TRÍ NGHI NGỜ]:")
                lines += impl_premises
        else:
            lines += [
                "[LỖI TỪ Z3 SOLVER]: General Error.",
                f"[CHI TIẾT]: {error_msg}",
                "[HƯỚNG DẪN]: Xem lại toàn bộ script, đặc biệt phần khai báo và assert.",
            ]

    elif status == "unsat" and unsat_core:
        core_str = "\n".join(f"   - {c}" for c in unsat_core[:3])
        prem_hints = "\n".join(
            f"   - Tiền đề {i+1}: {nl[:80]}"
            for i, nl in enumerate(original_nl)
        )
        lines += [
            "[LỖI TỪ Z3 SOLVER]: Contradiction in Premises (Mâu thuẫn tiền đề).",
            f"[CHI TIẾT - UNSAT CORE]:\n{core_str}",
            "[HƯỚNG DẪN]: Kiểm tra lại chiều implication của từng tiền đề:",
            prem_hints,
            "   - `A kéo theo B` phải là `(=> A B)`, KHÔNG đảo ngược.",
        ]

    elif status == "unknown":
        lines += [
            "[LỖI TỪ Z3 SOLVER]: Timeout / Undecidable.",
            "[HƯỚNG DẪN]: Đơn giản hóa script. Bỏ khai báo thừa. Dùng `(set-logic QF_UF)` nếu không cần quantifiers.",
        ]

    # ── Đính kèm script hiện tại để Qwen biết cần SEARCH đoạn nào ──
    lines += [
        "",
        "[SCRIPT HIỆN TẠI — dùng để xác định đoạn cần SEARCH]:",
        "```smtlib2",
        current_smtlib_script.strip(),
        "```",
        "",
        "Hãy xuất ra khối <SEARCH> và <REPLACE> để vá đúng lỗi trên.",
        "Chỉ sửa phần bị lỗi. Giữ nguyên khoảng trắng/tab trong <SEARCH>.",
        "Nếu đã thử sửa phần đó và vẫn thất bại (xem LỊCH SỬ ở trên), "
        "hãy SEARCH/REPLACE một đoạn HOÀN TOÀN KHÁC.",
    ]
    return "\n".join(lines)


def _normalize_for_matching(text: str) -> str:
    """
    v6 FIX #3: Chuẩn hóa whitespace 3 tầng để so sánh mờ (fuzzy match).

    Vấn đề gốc: apply_search_replace_patch chỉ strip trailing space.
    Không xử lý được:
      - Mixed indentation: Qwen dùng spaces, script gốc dùng tabs (hoặc ngược lại)
      - CRLF (\r\n) vs LF (\n) — phổ biến khi copy từ Windows editor
      - Multiple spaces/tabs giữa tokens SMT-LIB2

    Giải pháp: normalize trước khi so sánh, dùng vị trí match để
    extract đoạn gốc, rồi thực hiện replace trên chuỗi gốc.
    """
    # Tầng 1: chuẩn hóa line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    # Tầng 2: strip trailing whitespace từng dòng
    lines = [line.rstrip() for line in text.split("\n")]
    # Tầng 3: chuẩn hóa indentation — thay tabs bằng 2 spaces
    lines = [line.replace("\t", "  ") for line in lines]
    return "\n".join(lines)


def _find_and_replace_normalized(original: str, search: str, replace: str) -> Optional[str]:
    """
    Tìm `search` trong `original` sau khi cả hai đã được normalize.
    Nếu tìm thấy, thực hiện replace trên chuỗi gốc (không normalize)
    để giữ nguyên định dạng của phần không bị thay thế.
    """
    norm_orig   = _normalize_for_matching(original)
    norm_search = _normalize_for_matching(search)

    idx = norm_orig.find(norm_search)
    if idx == -1:
        return None

    # Đếm số dòng trước vị trí match để xác định range trong original
    lines_before = norm_orig[:idx].count("\n")
    lines_search  = norm_search.count("\n") + 1

    orig_lines = original.split("\n")
    # Lấy đúng số dòng tương ứng từ original để replace
    original_chunk = "\n".join(orig_lines[lines_before: lines_before + lines_search])
    patched = original.replace(original_chunk, replace, 1)
    return patched


def apply_search_replace_patch(original_script: str, patch_output: str) -> Optional[str]:
    """
    v6 — Thuật toán Hợp nhất (Merge Algorithm) nâng cấp:
    Trích xuất <SEARCH>...</SEARCH> và <REPLACE>...</REPLACE> từ output Qwen,
    thực hiện match 3 tầng và trả về script đã vá.

    Thứ tự ưu tiên:
      1. Exact match  — nhanh nhất, chính xác nhất
      2. Strip-trailing-space match — xử lý trailing whitespace
      3. Full normalize match — xử lý mixed indent + CRLF (v6 mới)
    """
    search_match  = re.search(r"<SEARCH>\s*(.*?)\s*</SEARCH>",  patch_output, re.DOTALL)
    replace_match = re.search(r"<REPLACE>\s*(.*?)\s*</REPLACE>", patch_output, re.DOTALL)

    if not search_match or not replace_match:
        print("    ⚠️  Patch parse failed: không tìm thấy <SEARCH>/<REPLACE> tags")
        return None

    search_block  = search_match.group(1)
    replace_block = replace_match.group(1)

    # ── Tầng 1: Exact match ──
    if search_block in original_script:
        patched = original_script.replace(search_block, replace_block, 1)
        print("    ✅ Patch applied (exact match)")
        return patched

    # ── Tầng 2: Strip trailing space (legacy fallback) ──
    stripped_search = "\n".join(line.rstrip() for line in search_block.splitlines())
    stripped_orig   = "\n".join(line.rstrip() for line in original_script.splitlines())
    if stripped_search in stripped_orig:
        patched = stripped_orig.replace(stripped_search, replace_block, 1)
        print("    ✅ Patch applied (trailing-space-stripped fallback)")
        return patched

    # ── Tầng 3: Full normalize (v6 mới — mixed indent + CRLF) ──
    patched = _find_and_replace_normalized(original_script, search_block, replace_block)
    if patched is not None:
        print("    ✅ Patch applied (full-normalize fallback: mixed indent/CRLF)")
        return patched

    print(f"    ⚠️  SEARCH block not found in script after 3-tier matching (len={len(search_block)})")
    return None


def strip_apology(text: str) -> str:
    """
    Lệnh cấm 'Xin lỗi' (The Apology Ban):
    Xóa sạch câu xin lỗi/giải thích ở đầu output trước khi parse SEARCH/REPLACE.
    Các model Instruct như Qwen3 có bản năng xin lỗi — hàm này vô hiệu hóa nó.
    """
    apology_patterns = [
        r"^(Tôi\s+(vô cùng\s+)?xin lỗi[^\n]*\n)+",
        r"^(I\s+apologize[^\n]*\n)+",
        r"^(Sorry[^\n]*\n)+",
        r"^(Xin lỗi[^\n]*\n)+",
        r"^([^<]*?)(Dưới đây là bản vá[^\n]*\n)",  # "Dưới đây là..." preamble
        r"^([^<]*?)(Here is the patch[^\n]*\n)",
    ]
    result = text.strip()
    for pat in apology_patterns:
        result = re.sub(pat, "", result, flags=re.IGNORECASE | re.MULTILINE).strip()
    return result


# ── Test module ──
_test_patch_output = """
Tôi vô cùng xin lỗi vì sự nhầm lẫn này. Dưới đây là bản vá lỗi:
<SEARCH>
(assert (is_prerequisite student_A CS101))
</SEARCH>
<REPLACE>
(assert (has_passed student_A CS101))
</REPLACE>
"""
_test_script = "(assert p)\n(assert (is_prerequisite student_A CS101))\n(check-sat)"
_cleaned     = strip_apology(_test_patch_output)
_patched     = apply_search_replace_patch(_test_script, _cleaned)
print("--- SEARCH/REPLACE Patch test (Tầng 1: exact match) ---")
print(f"Input script  : {_test_script}")
print(f"Apology strip : {repr(_cleaned[:60])}...")
print(f"Patched script: {_patched}")
assert _patched and "has_passed" in _patched and "is_prerequisite" not in _patched, "Patch test T1 failed!"
print("✅ Tầng 1 passed")

# Test tầng 3: mixed indentation (tab vs space)
_test_script_tab = "(assert p)\n\t(assert (is_prerequisite student_A CS101))\n(check-sat)"
_search_spaces   = "(assert (is_prerequisite student_A CS101))"  # Qwen dùng spaces, script dùng tab
_patch_tab = f"<SEARCH>\n{_search_spaces}\n</SEARCH>\n<REPLACE>\n(assert (has_passed student_A CS101))\n</REPLACE>"
_patched_tab = apply_search_replace_patch(_test_script_tab, _patch_tab)
assert _patched_tab and "has_passed" in _patched_tab, "Patch test T3 (mixed indent) failed!"
print("✅ Tầng 3 (mixed indent) passed")

print("✅ v6 Chiến lược 3: Search/Replace Patch module ready (3-tier whitespace normalizer)")


    ✅ Patch applied (exact match)
--- SEARCH/REPLACE Patch test (Tầng 1: exact match) ---
Input script  : (assert p)
(assert (is_prerequisite student_A CS101))
(check-sat)
Apology strip : '<SEARCH>\n(assert (is_prerequisite student_A CS101))\n</SEARCH'...
Patched script: (assert p)
(assert (has_passed student_A CS101))
(check-sat)
✅ Tầng 1 passed
    ✅ Patch applied (exact match)
✅ Tầng 3 (mixed indent) passed
✅ v6 Chiến lược 3: Search/Replace Patch module ready (3-tier whitespace normalizer)


---
## ⚙️ Chiến lược 4 — Early Stopping & Fallback (Max 3 vòng lặp)

Pipeline lặp tối đa **3 lần**. Nếu Z3 không hội tụ → fallback **"Unknown"**.

In [38]:
def predict_with_z3_loop(
    model,
    tokenizer,
    fol_list: List[str],
    nl_list: List[str],
    question: str,
    max_new_tokens: int = 3072,
    temperature: float = 0.0,
    max_iterations: int = MAX_Z3_ITERATIONS,
) -> Dict[str, Any]:
    messages = [
        {"role": "system", "content": SMTLIB_SYSTEM_PROMPT},  # BUG#6 FIX: dùng SMTLIB_SYSTEM_PROMPT, không phải SYSTEM_PROMPT chung
        {"role": "user",   "content": build_smtlib_prompt(fol_list, nl_list, question)},
    ]

    last_z3_result        = None
    current_smtlib_script = ""
    answer_history        = []
    failed_history        = []

    q_lower = question.lower().strip()
    is_mcq  = any(opt in q_lower for opt in ["(a)", "(b)", "(c)", "(d)", "option a", "option b"])

    for iteration in range(1, max_iterations + 1):
        print(f"  ↵ Iter {iteration}/{max_iterations}...", end=" ")

        # FIX THINKING MODE: enable_thinking=False tắt <think> blocks cho Qwen3
        try:
            input_ids = tokenizer.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=True,
                return_tensors="pt", enable_thinking=False,
            ).to(model.device)
        except TypeError:
            input_ids = tokenizer.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",
            ).to(model.device)

        # Dynamic Adapter Switching: Trả về Base Model để sinh SMT-LIB2 JSON
        with model.disable_adapter():
            with torch.no_grad():
                output_ids = model.generate(
                    input_ids,
                    max_new_tokens     = max_new_tokens if iteration == 1 else 1024,  # BUG#1 FIX: 400→600 cho iter 2+
                    max_length         = None,   # BUG#2 FIX: tránh xung đột với model config max_length
                    temperature        = temperature,
                    do_sample          = temperature > 0,
                    repetition_penalty = 1.0,
                    pad_token_id       = tokenizer.eos_token_id,
                    eos_token_id       = tokenizer.eos_token_id,
                )

        new_tokens = output_ids[0][input_ids.shape[1]:]
        raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True)
        # FIX THINKING MODE: strip thinking tags sau decode (phòng ngừa)
        raw_output = re.sub(r"<think>.*?</think>", "", raw_output, flags=re.DOTALL | re.IGNORECASE).strip()


        # Nếu chưa có script gốc (do Turn 1 lỗi JSON), ta tiếp tục cố gắng parse JSON
        if not current_smtlib_script:
            smtlib_json = validate_smtlib_json(raw_output)

            if smtlib_json is not None:
                current_smtlib_script = smtlib_json.get("goal", {}).get("smtlib2_script", "")

                if Z3_AVAILABLE:
                    if is_mcq:
                        goal_data = smtlib_json.get("goal", {})
                        option_asserts = goal_data.get("option_asserts", {})
                        prem_script = goal_data.get("premises_only_script", current_smtlib_script)
                        if option_asserts and len(option_asserts) >= 2:
                            mcq_result = run_z3_mcq_multigoal(prem_script, option_asserts)
                            if mcq_result["answer"] is not None:
                                return {"answer": mcq_result["answer"], "iterations": iteration, "z3_status": "mcq_multigoal", "smtlib_script": prem_script, "reasoning": raw_output}

                    goal_obj = smtlib_json.get("goal", {})
                    if "hypothesis_assert" in goal_obj and "premises_only_script" in goal_obj:
                        indep_ans = run_z3_check_independence(goal_obj["premises_only_script"], goal_obj["hypothesis_assert"])
                        if indep_ans:
                             return {"answer": indep_ans, "iterations": iteration, "z3_status": "independence_check", "smtlib_script": current_smtlib_script, "reasoning": raw_output}

                    z3_result = run_z3_on_smtlib(current_smtlib_script)
                else:
                    return {"answer": extract_final_answer(raw_output), "iterations": iteration, "z3_status": "no_z3", "smtlib_script": current_smtlib_script, "reasoning": raw_output}
            else:
                z3_result = {"status": "error", "error": "Invalid JSON in output", "model": None}
        else:
            # Đã có script gốc, thực hiện Patch Search/Replace
            cleaned_output = strip_apology(raw_output)
            patched_script = apply_search_replace_patch(current_smtlib_script, cleaned_output)

            if patched_script is None:
                # Nếu patch thất bại, ta ghi nhận lỗi nhưng cho phép loop tiếp để thử lại
                z3_result = {"status": "error", "error": "Patching failed: SEARCH block not found", "model": None}
            else:
                current_smtlib_script = patched_script
                z3_result = run_z3_on_smtlib(current_smtlib_script) if Z3_AVAILABLE else {"status": "error", "error": "Z3 not available"}

        last_z3_result = z3_result
        answer_from_z3 = interpret_z3_result(z3_result, question)
        answer_history.append({"iteration": iteration, "z3_status": z3_result.get("status"), "answer": answer_from_z3})

        print(f"Z3={z3_result.get('status')}", end=" ")

        if z3_result.get("status") in ("sat", "unsat"):
            return {"answer": answer_from_z3 or FALLBACK_ANSWER, "iterations": iteration, "z3_status": z3_result["status"], "smtlib_script": current_smtlib_script, "reasoning": raw_output}

        if iteration < max_iterations:
            messages.append({"role": "assistant", "content": raw_output})

            if not current_smtlib_script:
                # Case: Turn 1 failed JSON -> Yêu cầu sinh lại toàn bộ JSON thay vì Patch
                feedback = "[LỖI]: Output của bạn không phải là JSON hợp lệ. Vui lòng sinh lại TOÀN BỘ JSON object đúng schema."
            else:
                if iteration == 1: messages[0] = {"role": "system", "content": DEBUGGER_SYSTEM_PROMPT}
                # BUG#9 FIX: lưu nội dung SEARCH block thực tế thay vì hardcode "(attempt)"
                _search_match = re.search(r"<SEARCH>\s*(.*?)\s*</SEARCH>", raw_output, re.DOTALL)
                _patch_summary = _search_match.group(1)[:80].replace("\n", " ") if _search_match else "(no SEARCH block found)"
                failed_history.append({"iteration": iteration, "error": z3_result.get("error", "unknown"), "patch_summary": _patch_summary})
                feedback = format_z3_feedback_for_search_replace(z3_result, current_smtlib_script, iteration, fol_list, nl_list, failed_history=failed_history)

            messages.append({"role": "user", "content": feedback})
            print("→ Retrying...")

    return {"answer": FALLBACK_ANSWER, "iterations": max_iterations, "z3_status": last_z3_result.get("status", "error") if last_z3_result else "error", "smtlib_script": current_smtlib_script, "reasoning": "Max iterations reached", "answer_history": answer_history}

---
## 🔗 Pipeline Tích hợp — Evaluation v2 với 4 Chiến lược

In [8]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data: List[Dict] = json.load(f)

print(f"📦 Tổng số indices: {len(raw_data)}")
print("\n--- Cấu trúc 1 index mẫu ---")
sample = raw_data[0]
print(f"  idx            : {sample['idx']}")
print(f"  premises-FOL   : {len(sample['premises-FOL'])} premises")
print(f"  premises-NL    : {len(sample['premises-NL'])} premises")
print(f"  questions      : {len(sample['questions'])} questions")
print(f"  answers        : {sample['answers']}")

all_answers = []
for item in raw_data:
    all_answers.extend(item["answers"])

answer_dist = Counter(all_answers)
total_ans = sum(answer_dist.values())
print(f"\n📊 Phân phối nhãn (tổng {total_ans} mẫu sau unpack):")
for label, count in sorted(answer_dist.items()):
    pct = count / total_ans * 100
    bar = "█" * int(pct / 2)
    print(f"  {label:10s}: {count:4d} ({pct:5.1f}%) {bar}")

📦 Tổng số indices: 411

--- Cấu trúc 1 index mẫu ---
  idx            : [[6, 8, 12, 16], [2, 4, 5, 15]]
  premises-FOL   : 16 premises
  premises-NL    : 16 premises
  questions      : 2 questions
  answers        : ['Unknown', 'Yes']

📊 Phân phối nhãn (tổng 812 mẫu sau unpack):
  A         :  161 ( 19.8%) █████████
  B         :   43 (  5.3%) ██
  C         :   37 (  4.6%) ██
  D         :   22 (  2.7%) █
  No        :  156 ( 19.2%) █████████
  Unknown   :  150 ( 18.5%) █████████
  Yes       :  243 ( 29.9%) ██████████████


---
## 🗃️ Mistake-Driven Dataset — Recovery Dataset Generator

Tạo tập dữ liệu multi-turn **học từ sai lầm** (offline, trước khi train):

| Bước | Mô tả |
|------|-------|
| **1. Generate Bad Samples** | Chạy inference với `temperature=0.7` để ép Qwen sinh lỗi |
| **2. Capture Z3 Feedback** | Đưa output lỗi qua Z3, bắt error log / UNSAT |
| **3. Generate Golden Patches** | Dùng Teacher Model sinh `<SEARCH>/<REPLACE>` chuẩn |

Cấu trúc ChatML multi-turn:  
`user(T1) → assistant(T1: lỗi) → user(T2: Z3 feedback) → assistant(T2: patch)`

In [39]:
# ============================================================
#  MISTAKE-DRIVEN DATASET GENERATOR (Offline — chạy trước Train)
# ============================================================
# Quy trình 3 bước tạo tập dữ liệu sửa lỗi:
#   Bước 1: Generate Bad Samples (temperature=0.7 ép lỗi)
#   Bước 2: Capture Z3 Feedback
#   Bước 3: Ghép thành multi-turn ChatML mẫu cho fine-tuning

RECOVERY_DATASET_TEMPERATURE = 0.7   # Cố tình tạo output lỗi
RECOVERY_DATASET_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "recovery_dataset.json")


def generate_bad_sample(
    model,
    tokenizer,
    fol_list: List[str],
    nl_list: List[str],
    question: str,
    max_new_tokens: int = 400,
) -> str:
    """
    Bước 1: Chạy inference với temperature cao để ép Qwen sinh SMT-LIB2 bị lỗi.
    Trả về raw output (có thể chứa JSON hợp lệ hoặc bị lỗi).
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": build_smtlib_prompt(fol_list, nl_list, question)},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens     = max_new_tokens,
            temperature        = RECOVERY_DATASET_TEMPERATURE,
            do_sample          = True,   # Bật sampling để tạo đa dạng lỗi
            repetition_penalty = 1.0,
            pad_token_id       = tokenizer.eos_token_id,
            eos_token_id       = tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def capture_z3_error(bad_output: str) -> Tuple[Optional[str], Dict[str, Any]]:
    """
    Bước 2: Đưa output lỗi qua Z3, bắt error log / UNSAT.
    Trả về (smtlib_script, z3_result).
    smtlib_script=None nếu không parse được JSON.
    """
    smtlib_json = validate_smtlib_json(bad_output)
    if smtlib_json is None or smtlib_json.get("_unsolvable"):
        return None, {"status": "error", "error": "Invalid JSON output", "model": None}

    script = smtlib_json.get("goal", {}).get("smtlib2_script", "")
    if not script:
        return None, {"status": "error", "error": "Missing smtlib2_script in goal", "model": None}

    z3_result = run_z3_on_smtlib(script)
    return script, z3_result


def build_recovery_chatml_sample(
    fol_list: List[str],
    nl_list: List[str],
    question: str,
    bad_output: str,          # Assistant Turn 1: output lỗi
    z3_result: Dict,
    smtlib_script: str,
    golden_patch: str,        # Assistant Turn 2: SEARCH/REPLACE chuẩn (từ Teacher Model)
) -> Dict[str, Any]:
    """
    Bước 3: Ghép thành mẫu ChatML multi-turn chuẩn cho fine-tuning.

    Cấu trúc:
      system  → SYSTEM_PROMPT
      user    → câu hỏi gốc (Turn 1)
      assistant → output lỗi của Qwen (Turn 1) — model học cách dừng ở đây nếu không chắc
      user    → Z3 feedback + script (Turn 2)
      assistant → SEARCH/REPLACE patch chuẩn (Turn 2) — model học cách vá cục bộ
    """
    feedback_prompt = format_z3_feedback_for_search_replace(
        z3_result, smtlib_script, iteration=1,
        original_fol=fol_list, original_nl=nl_list,
    )
    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": build_smtlib_prompt(fol_list, nl_list, question)},
            {"role": "assistant", "content": bad_output},        # Turn 1: có lỗi
            {"role": "user",      "content": feedback_prompt},   # Turn 2: Z3 feedback
            {"role": "assistant", "content": golden_patch},      # Turn 2: bản vá chuẩn
        ],
        "answer_label":    None,   # Sẽ không dùng trực tiếp cho eval
        "is_recovery":     True,
        "z3_error_status": z3_result.get("status"),
    }


def run_recovery_dataset_generation(
    raw_data: List[Dict],
    model,
    tokenizer,
    max_samples: int = 100,
    seed: int = SEED,
) -> List[Dict]:
    """
    Pipeline tổng: chạy Bước 1 + 2 trên tối đa max_samples entries.
    Bước 3 (golden_patch) cần Teacher Model (GPT-4o / Claude) offline —
    hàm này tạo placeholder để điền sau.

    Trả về danh sách dict gồm: entry gốc + bad_output + z3_result + script
    để xuất ra file cho Teacher Model điền golden_patch.
    """
    rng = random.Random(seed)
    candidates = [(entry, q, a, exp)
                  for entry in raw_data
                  for q, a, exp in zip(entry["questions"], entry["answers"], entry["explanation"])]
    rng.shuffle(candidates)
    candidates = candidates[:max_samples]

    recovery_samples = []
    error_count      = 0

    print(f"🔍 Generating bad samples (temperature={RECOVERY_DATASET_TEMPERATURE})...")
    for idx, (entry, question, expected_ans, _) in enumerate(candidates):
        fol_list = entry["premises-FOL"]
        nl_list  = entry["premises-NL"]

        # Bước 1: sinh output lỗi
        bad_output = generate_bad_sample(model, tokenizer, fol_list, nl_list, question)

        # Bước 2: bắt lỗi Z3
        smtlib_script, z3_result = capture_z3_error(bad_output)

        # Chỉ giữ lại mẫu thực sự bị lỗi (status=error hoặc unsat với unsat_core)
        is_bad = (
            z3_result["status"] == "error"
            or (z3_result["status"] == "unsat" and z3_result.get("unsat_core"))
        )
        if not is_bad:
            continue

        error_count += 1
        recovery_samples.append({
            "fol_list":      fol_list,
            "nl_list":       nl_list,
            "question":      question,
            "expected_ans":  expected_ans,
            "bad_output":    bad_output,          # Cần điền golden_patch sau
            "smtlib_script": smtlib_script or "",
            "z3_result":     z3_result,
            "golden_patch":  None,                 # Placeholder — Teacher Model điền
        })

        if (idx + 1) % 20 == 0:
            print(f"  [{idx+1}/{len(candidates)}] Error samples collected: {error_count}")

    # Lưu ra file để Teacher Model (GPT-4o/Claude) điền golden_patch offline
    with open(RECOVERY_DATASET_OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(recovery_samples, f, ensure_ascii=False, indent=2)

    print(f"\n✅ Recovery dataset generation complete.")
    print(f"   Total candidates  : {len(candidates)}")
    print(f"   Error samples     : {error_count} ({error_count/len(candidates)*100:.1f}%)")
    print(f"   Saved to          : {RECOVERY_DATASET_OUTPUT_PATH}")
    print(f"   ⚠️  Bước tiếp theo : Dùng Teacher Model (GPT-4o / Claude) điền golden_patch")
    print(f"      vào từng entry rồi gọi build_recovery_chatml_sample() để tạo tập train.")
    return recovery_samples


# ── Tích hợp vào dataset nếu recovery_dataset.json đã có golden_patch ──
def load_and_merge_recovery_dataset(
    base_samples: List[Dict],
    recovery_path: str = RECOVERY_DATASET_OUTPUT_PATH,
) -> List[Dict]:
    """
    Sau khi Teacher Model đã điền golden_patch:
    Load file, build_recovery_chatml_sample() và merge vào base_samples.
    """
    if not os.path.exists(recovery_path):
        print(f"ℹ️  {recovery_path} chưa tồn tại — bỏ qua merge.")
        return base_samples

    with open(recovery_path, encoding="utf-8") as f:
        recovery_raw = json.load(f)

    merged      = []
    skip_count  = 0
    for r in recovery_raw:
        if r.get("golden_patch") is None:
            skip_count += 1
            continue
        sample = build_recovery_chatml_sample(
            fol_list      = r["fol_list"],
            nl_list       = r["nl_list"],
            question      = r["question"],
            bad_output    = r["bad_output"],
            z3_result     = r["z3_result"],
            smtlib_script = r["smtlib_script"],
            golden_patch  = r["golden_patch"],
        )
        merged.append(sample)

    all_samples = base_samples + merged
    print(f"✅ Recovery dataset merged: +{len(merged)} multi-turn samples")
    if skip_count:
        print(f"   ⚠️  {skip_count} entries bỏ qua (golden_patch=None — chưa có Teacher annotation)")
    return all_samples


# ── Chạy thử (set RUN_RECOVERY=True để thực sự generate) ──
RUN_RECOVERY = False
if RUN_RECOVERY:
    recovery_samples = run_recovery_dataset_generation(
        raw_data, model, tokenizer, max_samples=100
    )
else:
    print("ℹ️  RUN_RECOVERY=False. Đặt True sau khi train xong lần đầu để tạo Recovery Dataset.")
    print("   Sau đó dùng Teacher Model điền golden_patch và gọi load_and_merge_recovery_dataset().")


ℹ️  RUN_RECOVERY=False. Đặt True sau khi train xong lần đầu để tạo Recovery Dataset.
   Sau đó dùng Teacher Model điền golden_patch và gọi load_and_merge_recovery_dataset().


In [31]:
SYSTEM_PROMPT = """You are a rigorous logical reasoning assistant specializing in First-Order Logic (FOL).
Given a set of premises (in natural language and FOL notation), you must:
1. Analyze the logical structure of each premise carefully.
2. Apply formal inference rules: modus ponens, contrapositive, universal instantiation, existential instantiation, hypothetical syllogism.
3. Reason step-by-step (Chain of Thought) BEFORE committing to a final answer.
4. For multiple-choice questions (A/B/C/D): evaluate each option against the premises.
5. For Yes/No questions: verify the statement logically.
6. If the premises are INSUFFICIENT to determine a unique conclusion, your Final Answer MUST be exactly: Unknown
7. Never hallucinate a conclusion that is not entailed by the premises.
Format your response EXACTLY as:
### Step-by-Step Reasoning
<your reasoning here>
### Final Answer
<A or B or C or D or Yes or No or Unknown>"""

# BUG #6 FIX: SMTLIB_SYSTEM_PROMPT riêng cho predict_with_z3_loop
# Tránh mâu thuẫn: SYSTEM_PROMPT yêu cầu "### Final Answer" nhưng
# build_smtlib_prompt yêu cầu "Return ONLY a valid JSON object"
SMTLIB_SYSTEM_PROMPT = """You are an expert SMT-LIB2 translator.
Your ONLY task is to translate logical premises and a hypothesis into a valid JSON object.
Use the "thought_process" field inside the JSON to write your reasoning — do not output any text outside the JSON.
Output MUST be a single raw JSON object starting with { and ending with }. No text before or after."""


# v5: Prompt bổ sung cho SMT-LIB2 JSON — yêu cầu Qwen sinh thêm
# option_asserts (cho MCQ) và premises_only_script (tách riêng premises)
SMTLIB_EXTRA_INSTRUCTIONS = """
For MULTIPLE-CHOICE questions, the `goal` object MUST also include:
- "premises_only_script": SMT-LIB2 script with ONLY declare + assert premises (NO check-sat, NO hypothesis)
- "option_asserts": object with keys "A","B","C","D", each value is "(assert <FOL expression for that option>)"
  Example: {"A": "(assert (Passed Alice CS101))", "B": "(assert (not (Passed Alice CS101)))", ...}
  The backend will use Z3 push()/pop() to verify each option independently.

For YES/NO questions, the `goal` object MUST also include:
- "premises_only_script": SMT-LIB2 with only premises (no hypothesis, no check-sat)
- "hypothesis_assert": "(assert <the hypothesis to test>)" — just one assert statement
  The backend will call Z3 twice: once for hypothesis, once for its negation.
"""


def format_premises(fol_list: List[str], nl_list: List[str]) -> str:
    lines = ["### Premises"]
    for i, (fol, nl) in enumerate(zip(fol_list, nl_list), 1):
        lines.append(f"P{i}. [NL]  {nl}")
        lines.append(f"   [FOL] {fol}")
    return "\n".join(lines)


def build_user_message(fol_list: List[str], nl_list: List[str], question: str) -> str:
    premises_block = format_premises(fol_list, nl_list)
    return f"{premises_block}\n\n### Question\n{question}"


def build_assistant_message(explanation: str, answer: str) -> str:
    """Answer-Last: CoT trước, đáp án sau để model học suy luận."""
    return (
        f"### Step-by-Step Reasoning\n{explanation}\n\n"
        f"### Final Answer\n{answer}"
    )


print("✅ Prompt templates defined.")
s = raw_data[0]
print("\n--- User Message preview ---")
print(build_user_message(s["premises-FOL"], s["premises-NL"], s["questions"][0])[:400])
print("\n--- Assistant Message preview ---")
print(build_assistant_message(s["explanation"][0], s["answers"][0])[:300])

✅ Prompt templates defined.

--- User Message preview ---
### Premises
P1. [NL]  All students must enroll in at least one core subject.
   [FOL] ∀x (EnrollsCoreSubject(x))
P2. [NL]  If a student attends all lectures, they will have a higher chance of passing the course.
   [FOL] ∀x (AttendsLectures(x) → HigherChancePass(x))
P3. [NL]  If a student does not attend all lectures, they might struggle to understand key concepts.
   [FOL] ∀x (¬AttendsLectures(x

--- Assistant Message preview ---
### Step-by-Step Reasoning
From premise 6 (There exists at least one student who participates in an academic competition), premise 8 (If a student joins the research group, they must contribute to at least one published paper), premise 12 (If joining the research group requires contributing to a pub


In [11]:
def unpack_dataset(raw_data: List[Dict]) -> List[Dict]:
    """Mỗi (index × question) → 1 training sample độc lập."""
    samples = []
    for entry in raw_data:
        fol_list     = entry["premises-FOL"]
        nl_list      = entry["premises-NL"]
        questions    = entry["questions"]
        answers      = entry["answers"]
        explanations = entry["explanation"]

        assert len(questions) == len(answers) == len(explanations), (
            f"Mismatch in entry idx={entry.get('idx')}: "
            f"q={len(questions)}, a={len(answers)}, exp={len(explanations)}"
        )

        for q, a, exp in zip(questions, answers, explanations):
            user_msg      = build_user_message(fol_list, nl_list, q)
            assistant_msg = build_assistant_message(exp, a)
            sample = {
                "messages": [
                    {"role": "system",    "content": SYSTEM_PROMPT},
                    {"role": "user",      "content": user_msg},
                    {"role": "assistant", "content": assistant_msg},
                ],
                "answer_label": a,
                "is_unknown":   a.strip().lower() == "unknown",
                # Lưu gốc để evaluate toàn bộ 411 indices sau
                "_raw_entry_idx": id(entry),
            }
            samples.append(sample)
    return samples


flat_samples = unpack_dataset(raw_data)
unknown_count = sum(1 for s in flat_samples if s["is_unknown"])
print(f"📦 Tổng samples sau unpack  : {len(flat_samples)}")
print(f"   Nhãn Unknown             : {unknown_count} ({unknown_count/len(flat_samples)*100:.1f}%)")
print(f"   Nhãn có đáp án           : {len(flat_samples) - unknown_count}")

📦 Tổng samples sau unpack  : 812
   Nhãn Unknown             : 150 (18.5%)
   Nhãn có đáp án           : 662


In [12]:
# ============================================================
#  v5: XÓA OVERSAMPLING — Chống Lazy Agent Bias
# ============================================================
# v4 dùng apply_adversarial_oversampling() nhân bản nhãn Unknown 3x.
# Điều này tạo ra "Lazy Agent Bias": mô hình học shortcut
#   is_solvable=False → Unknown  (40% hit rate mà không cần suy luận)
# thay vì học cách chứng minh thật sự tại sao thiếu dữ kiện.
#
# v5 FIX — 2 tầng phòng thủ:
#   Tầng 1 (Training): Không oversampling. Dùng flat_samples trực tiếp.
#                      CoT trong build_assistant_message() ép mô hình
#                      phải giải thích lý do Unknown (không thể học vẹt).
#   Tầng 2 (Inference): Z3 tự chứng minh Unknown bằng toán học:
#                       SAT(H) ∧ SAT(¬H) → độc lập logic → "Unknown"
#                       LLM không còn quyền tự khai báo is_solvable=False.

# Không gọi oversampling — dùng thẳng flat_samples đã unpack
augmented_samples = flat_samples   # Tên giữ nguyên để các cell sau không bị break

unknown_count = sum(1 for s in augmented_samples if s["is_unknown"])
known_count   = len(augmented_samples) - unknown_count
print(f"✅ v5: Không oversampling — phân phối nhãn tự nhiên:")
print(f"   Tổng samples  : {len(augmented_samples)}")
print(f"   Unknown       : {unknown_count} ({unknown_count/len(augmented_samples)*100:.1f}%)")
print(f"   Known answers : {known_count} ({known_count/len(augmented_samples)*100:.1f}%)")
print(f"   ℹ️  Z3 sẽ chứng minh Unknown bằng SAT(H)∧SAT(¬H) tại inference time.")


✅ v5: Không oversampling — phân phối nhãn tự nhiên:
   Tổng samples  : 812
   Unknown       : 150 (18.5%)
   Known answers : 662 (81.5%)
   ℹ️  Z3 sẽ chứng minh Unknown bằng SAT(H)∧SAT(¬H) tại inference time.


In [13]:
def split_dataset(
    samples: List[Dict],
    train_ratio: float = TRAIN_RATIO,
    val_ratio:   float = VAL_RATIO,
    seed: int = SEED,
) -> DatasetDict:
    shuffled = copy.deepcopy(samples)
    random.Random(seed).shuffle(shuffled)

    n = len(shuffled)
    n_train = int(n * train_ratio)
    n_val   = int(n * val_ratio)

    train_list = shuffled[:n_train]
    val_list   = shuffled[n_train : n_train + n_val]
    test_list  = shuffled[n_train + n_val :]

    def to_hf_dataset(lst):
        return Dataset.from_list([{"messages": s["messages"]} for s in lst])

    ds = DatasetDict({
        "train":      to_hf_dataset(train_list),
        "validation": to_hf_dataset(val_list),
        "test":       to_hf_dataset(test_list),
    })

    print("📊 Dataset splits:")
    for split_name, split_ds in ds.items():
        print(f"   {split_name:12s}: {len(split_ds):5d} samples")
    return ds


dataset = split_dataset(augmented_samples)

📊 Dataset splits:
   train       :   690 samples
   validation  :    81 samples
   test        :    41 samples


In [14]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)

# FIX #3: Qwen3 dùng chat template "qwen-2.5" (tương thích Qwen3)
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

print(f"✅ Model loaded: {MODEL_NAME}")
print(f"   Vocab size    : {tokenizer.vocab_size}")
print(f"   Max seq length: {MAX_SEQ_LEN}")

==((====))==  Unsloth 2026.5.8: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/6.07G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/4.67k [00:00<?, ?B/s]

unsloth/Qwen3-8B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ Model loaded: unsloth/Qwen3-8B-bnb-4bit
   Vocab size    : 151643
   Max seq length: 4096


In [15]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_R,
    target_modules             = LORA_TARGET_MODULES,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = SEED,
    use_rslora                 = False,
    loftq_config               = None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA applied:")
print(f"   Trainable params: {trainable:,} ({trainable/total_p*100:.2f}%)")
print(f"   Total params    : {total_p:,}")
print(f"   Target modules  : {LORA_TARGET_MODULES}")

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.5.8 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✅ LoRA applied:
   Trainable params: 174,587,904 (3.57%)
   Total params    : 4,892,439,552
   Target modules  : ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']


In [16]:
# FIX #2 & #4: Chỉ định nghĩa 1 lần, đúng signature cho SFTTrainer
# SFTConfig với dataset_text_field="text" yêu cầu formatting_func trả về dict
# Nhưng khi dùng formatting_func KHÔNG có dataset_text_field → trả list of str

def formatting_prompts_func(examples):
    """
    Trả về list of strings (không dict).
    SFTTrainer với formatting_func sẽ tự tạo cột 'text' từ list này.
    """
    convos = examples["messages"]
    # BUG FIX: isinstance(convos[0], dict) luôn True vì message là dict
    # → không thể dùng để phân biệt single vs batch.
    # Batch: convos là List[List[Dict]] → convos[0] là List[Dict] → isinstance list
    # Single: convos là List[Dict] → convos[0] là Dict → isinstance dict
    if convos and isinstance(convos[0], dict):
        convos = [convos]  # Wrap single sample thành batch

    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize              = False,
            add_generation_prompt = False,
        )
        for convo in convos
    ]
    return texts   # list of str, không phải dict


# Verify formatting
sample_text = formatting_prompts_func({"messages": dataset["train"][0]["messages"]})
print("--- Formatted sample (đầu 600 ký tự) ---")
if isinstance(sample_text, list):
    print(sample_text[0][:600])
else:
    print(sample_text[:600])
print("\n[... truncated ...]")

--- Formatted sample (đầu 600 ký tự) ---
<|im_start|>system
You are a rigorous logical reasoning assistant specializing in First-Order Logic (FOL).
Given a set of premises (in natural language and FOL notation), you must:
1. Analyze the logical structure of each premise carefully.
2. Apply formal inference rules: modus ponens, contrapositive, universal instantiation, existential instantiation, hypothetical syllogism.
3. Reason step-by-step (Chain of Thought) BEFORE committing to a final answer.
4. For multiple-choice questions (A/B/C/D): evaluate each option against the premises.
5. For Yes/No questions: verify the statement logicall

[... truncated ...]


In [17]:
from transformers import TrainingArguments, TrainerCallback
from trl import SFTTrainer, SFTConfig
from tqdm.auto import tqdm
import torch

# ── Custom Progress Callback ───────────────────────────────────────
class TqdmProgressCallback(TrainerCallback):
    def __init__(self):
        self.pbar = None
    def on_train_begin(self, args, state, control, **kwargs):
        self.pbar = tqdm(total=state.max_steps, desc="🚀 Training",
                         unit="step", colour="green")
    def on_step_end(self, args, state, control, **kwargs):
        if self.pbar:
            self.pbar.update(1)
            if state.log_history:
                self.pbar.set_postfix({"loss": f"{state.log_history[-1].get('loss', 0):.4f}"})
    def on_train_end(self, args, state, control, **kwargs):
        if self.pbar:
            self.pbar.close()

# ── Áp dụng Chat Template trước khi train để tránh lỗi Jinja2 ──
def apply_template(example):
    example["text"] = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return example

train_ds = dataset["train"].map(apply_template)
eval_ds  = dataset["validation"].map(apply_template)

_steps_per_epoch = max(1, len(train_ds) // (BATCH_SIZE * GRAD_ACCUM_STEPS))
_logging_steps   = max(1, min(10, _steps_per_epoch // 10))
# FIX #5: Không dùng dataset_text_field khi đã dùng formatting_func
# FIX #6: Thêm các tham số ổn định training
training_args = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    # dataset_text_field KHÔNG set khi dùng formatting_func
    packing                     = False,
    padding_free                = False,
    assistant_only_loss         = False,  # Tắt để tránh xung đột với train_on_responses_only
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,
    num_train_epochs            = NUM_EPOCHS,
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = LR_SCHEDULER,
    warmup_ratio                = WARMUP_RATIO,
    weight_decay                = WEIGHT_DECAY,
    max_grad_norm               = MAX_GRAD_NORM,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 20,
    eval_strategy               = "steps",
    eval_steps                  = 100,
    save_strategy               = "steps",
    save_steps                  = 100,
    save_total_limit            = 3,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    seed                        = SEED,
    report_to                   = "none",
    dataloader_num_workers      = 2,
    optim                       = "adamw_8bit",
    push_to_hub                 = False,
)

trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = dataset["train"],
    eval_dataset     = dataset["validation"],
    formatting_func  = formatting_prompts_func,
    args             = training_args,
)

# FIX #3: Qwen3 tokens đúng cho train_on_responses_only
# Verify token markers trước khi dùng
sample_formatted = tokenizer.apply_chat_template(
    dataset["train"][0]["messages"],
    tokenize=False,
    add_generation_prompt=False,
)
print("Token markers check:")
print("  <|im_start|>user    present:", "<|im_start|>user" in sample_formatted)
print("  <|im_start|>assistant present:", "<|im_start|>assistant" in sample_formatted)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

print("\n✅ SFTTrainer initialized successfully!")
print(f"   Train samples : {len(dataset['train'])}")
print(f"   Val samples   : {len(dataset['validation'])}")

Map:   0%|          | 0/690 [00:00<?, ? examples/s]

Map:   0%|          | 0/81 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/690 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/81 [00:00<?, ? examples/s]

...

Map (num_proc=6):   0%|          | 0/690 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/690 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/81 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/81 [00:00<?, ? examples/s]


✅ SFTTrainer initialized successfully!
   Train samples : 690
   Val samples   : 81


In [18]:
print("🚀 Starting fine-tuning...")
print("=" * 60)

trainer_stats = trainer.train()

print("\n" + "=" * 60)
print("✅ Training complete!")
print(f"   Total steps    : {trainer_stats.global_step}")
print(f"   Training time  : {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"   Samples/second : {trainer_stats.metrics['train_samples_per_second']:.2f}")
print(f"   Final loss     : {trainer_stats.metrics['train_loss']:.4f}")

if torch.cuda.is_available():
    peak = torch.cuda.max_memory_allocated() / 1e9
    print(f"   Peak VRAM used : {peak:.2f} GB")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🚀 Starting fine-tuning...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 690 | Num Epochs = 4 | Total steps = 176
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 174,587,904 of 8,365,323,264 (2.09% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
100,0.107709,0.162691
176,0.031581,0.165040


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 


✅ Training complete!
   Total steps    : 176
   Training time  : 9865s
   Samples/second : 0.28
   Final loss     : 0.1884
   Peak VRAM used : 10.87 GB


In [19]:
LORA_SAVE_PATH = os.path.join(OUTPUT_DIR, "lora_adapter")
os.makedirs(LORA_SAVE_PATH, exist_ok=True)

model.save_pretrained(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)

saved_files = os.listdir(LORA_SAVE_PATH)
total_size_mb = sum(
    os.path.getsize(os.path.join(LORA_SAVE_PATH, f))
    for f in saved_files
) / 1e6

print("✅ LoRA adapter saved.")
print(f"   Path  : {LORA_SAVE_PATH}")
print(f"   Files : {saved_files}")
print(f"   Size  : {total_size_mb:.1f} MB")

Unsloth: Restored added_tokens_decoder metadata in /content/lora_output_qwen3/lora_adapter/tokenizer_config.json.


✅ LoRA adapter saved.
   Path  : /content/lora_output_qwen3/lora_adapter
   Files : ['README.md', 'tokenizer.json', 'chat_template.jinja', 'adapter_model.safetensors', 'tokenizer_config.json', 'adapter_config.json']
   Size  : 709.9 MB


In [28]:
import shutil
from google.colab import drive
import os

# 1. Kích hoạt quyền truy cập vào Google Drive
drive.mount('/content/drive')

# 2. Định nghĩa đường dẫn gốc (trong Colab) và đường dẫn đích (trên Drive)
source_checkpoint_dir = "/content/lora_output_qwen3/checkpoint-176"
destination_drive_dir = "/content/drive/MyDrive/Colab_Notebooks/AI/Thi/EXACT2026/Qwen3_8B"

# 3. Tiến hành sao chép nguyên vẹn cây thư mục (bao gồm tất cả các file cấu hình và trọng số)
# Sử dụng dirs_exist_ok=True để cho phép thư mục đích tồn tại. Các tệp tin trùng tên sẽ được ghi đè.
shutil.copytree(source_checkpoint_dir, destination_drive_dir, dirs_exist_ok=True)
print("✅ Hệ thống đã di dời toàn bộ tệp tin cấu hình và trọng số LoRA lên Drive an toàn!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Hệ thống đã di dời toàn bộ tệp tin cấu hình và trọng số LoRA lên Drive an toàn!


In [32]:
# Chuyển sang inference mode
FastLanguageModel.for_inference(model)


def predict(fol_list: List[str], nl_list: List[str], question: str,
            max_new_tokens: int = 512, temperature: float = 0.0) -> str:
    """Inference với repetition_penalty để tránh loop (FIX #8)."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": build_user_message(fol_list, nl_list, question)},
    ]

    # FIX THINKING MODE
    try:
        input_ids = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_tensors="pt", enable_thinking=False,
        ).to(model.device)
    except TypeError:
        input_ids = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",
        ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens      = max_new_tokens,
            max_length          = None,  # [FIX SỐ 1] Triệt tiêu xung đột chiều dài chuỗi
            temperature         = temperature,
            do_sample           = temperature > 0,
            repetition_penalty  = 1.0,    # v5: penalty=1.0 (tắt hoàn toàn) — SMT-LIB2 bắt buộc lặp token hợp lệ
            pad_token_id        = tokenizer.eos_token_id,
            eos_token_id        = tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][input_ids.shape[1]:]
    result = tokenizer.decode(new_tokens, skip_special_tokens=True)
    # FIX THINKING MODE
    result = re.sub(r"<think>.*?</think>", "", result, flags=re.DOTALL | re.IGNORECASE).strip()
    return result


print("✅ predict() function ready with repetition_penalty=1.0")

✅ predict() function ready with repetition_penalty=1.0


In [21]:
def extract_final_answer(model_output: str) -> str:
    """
    FIX #7: Regex mạnh hơn — nhiều pattern fallback để không miss.
    Priority order: Final Answer block > inline answer patterns.
    """
    text = model_output.strip()

    # Pattern 1: phần sau '### Final Answer'
    match = re.search(
        r"###\s*Final\s*Answer[:\s]*\n?\s*(.+)",
        text, re.IGNORECASE
    )
    if match:
        ans = match.group(1).strip().split("\n")[0].strip()
        if re.match(r"^unknown", ans, re.IGNORECASE):
            return "Unknown"
        m_letter = re.match(r"^([A-D])[.)\s:]", ans, re.IGNORECASE)
        if m_letter:
            return m_letter.group(1).upper()
        if re.match(r"^[A-D]$", ans, re.IGNORECASE):
            return ans.upper()
        # Yes/No
        if re.match(r"^yes", ans, re.IGNORECASE):
            return "Yes"
        if re.match(r"^no", ans, re.IGNORECASE):
            return "No"

    # Pattern 2: "Answer: X" hoặc "answer is X"
    match2 = re.search(
        r"(?:answer\s*(?:is|:)\s*)([A-D]|unknown|yes|no)",
        text, re.IGNORECASE
    )
    if match2:
        g = match2.group(1).strip()
        if g.lower() == "unknown": return "Unknown"
        if g.lower() == "yes":    return "Yes"
        if g.lower() == "no":     return "No"
        return g.upper()

    # Pattern 3: standalone letter near end of text
    last_500 = text[-500:]
    match3 = re.findall(r"\b([A-D])\b", last_500)
    if match3:
        return match3[-1].upper()

    # Pattern 4: Unknown/Yes/No anywhere
    if re.search(r"\bunknown\b", text, re.IGNORECASE):
        return "Unknown"
    if re.search(r"\byes\b", text, re.IGNORECASE):
        return "Yes"
    if re.search(r"\bno\b", text, re.IGNORECASE):
        return "No"

    return "UNPARSEABLE"


# Test
test_cases = [
    ("### Final Answer\nA", "A"),
    ("### Final Answer\nUnknown", "Unknown"),
    ("### Final Answer\nYes", "Yes"),
    ("some reasoning\n### Final Answer\nB. Because...", "B"),
    ("the answer is C", "C"),
    ("UNPARSEABLE output xyz", "UNPARSEABLE"),
]
print("Testing extract_final_answer:")
for text, expected in test_cases:
    got = extract_final_answer(text)
    status = "✅" if got == expected else "❌"
    print(f"  {status} '{text[:40]}'  →  got={got}, expected={expected}")

Testing extract_final_answer:
  ✅ '### Final Answer
A'  →  got=A, expected=A
  ✅ '### Final Answer
Unknown'  →  got=Unknown, expected=Unknown
  ✅ '### Final Answer
Yes'  →  got=Yes, expected=Yes
  ✅ 'some reasoning
### Final Answer
B. Becau'  →  got=B, expected=B
  ✅ 'the answer is C'  →  got=C, expected=C
  ✅ 'UNPARSEABLE output xyz'  →  got=UNPARSEABLE, expected=UNPARSEABLE


In [40]:
# ── DEBUG CELL: xem raw output thực tế của model ──
import re

_entry = raw_data[0]
_fol   = _entry["premises-FOL"]
_nl    = _entry["premises-NL"]
_q     = _entry["questions"][0]

_messages = [
    {"role": "system", "content": SMTLIB_SYSTEM_PROMPT},
    {"role": "user",   "content": build_smtlib_prompt(_fol, _nl, _q)},
]

# Thử apply_chat_template với enable_thinking=False
try:
    _ids = tokenizer.apply_chat_template(
        _messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", enable_thinking=False,
    ).to(model.device)
    print("✅ enable_thinking=False supported")
except TypeError as e:
    print(f"❌ enable_thinking not supported: {e}")
    _ids = tokenizer.apply_chat_template(
        _messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",
    ).to(model.device)

with model.disable_adapter():
    with torch.no_grad():
        _out = model.generate(
            _ids,
            max_new_tokens=3072,  # Tăng từ 1024 lên 2048 để tránh bị cut-off
            max_length=None,
            temperature=0.0,
            do_sample=False,
            repetition_penalty=1.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

_new = _out[0][_ids.shape[1]:]
_raw = tokenizer.decode(_new, skip_special_tokens=False)  # skip_special_tokens=FALSE để thấy tất cả

print("\n" + "="*60)
print("RAW OUTPUT (skip_special_tokens=False):")
print("="*60)
print(repr(_raw[:500]))   # repr để thấy \n, \t, escape chars
print("...")
print(repr(_raw[-200:]))

print("\n" + "="*60)
_raw_skip = tokenizer.decode(_new, skip_special_tokens=True)
print("RAW OUTPUT (skip_special_tokens=True):")
print("="*60)
print(repr(_raw_skip[:500]))

✅ enable_thinking=False supported


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



RAW OUTPUT (skip_special_tokens=False):
"<think>\nOkay, let's tackle this problem. The user wants me to translate the given premises and the question into SMT-LIB2 format. First, I need to understand the premises and the question thoroughly.\n\nLooking at the premises, there are 16 statements, each with a natural language description and a corresponding FOL formula. The question is a multiple-choice (A-D) asking which conclusion is correct based on the premises. The task is to translate all premises and the hypothesis into SMT-LIB2, inclu"
...
"r's example for the 'smtlib2_script' includes the premises and the negation of the hypothesis. So for the multiple-choice, the 'smtlib2_script' is the premises plus the negation of the hypothesis, and"

RAW OUTPUT (skip_special_tokens=True):
"<think>\nOkay, let's tackle this problem. The user wants me to translate the given premises and the question into SMT-LIB2 format. First, I need to understand the premises and the question thoroughly.\n\nL

In [41]:
# Test _strip_thinking
_cleaned = _strip_thinking(_raw_skip)
print("\nSau khi strip:")
print(repr(_cleaned[:300]))

# Test validate_smtlib_json
print("\nChạy validate_smtlib_json:")
_result = validate_smtlib_json(_raw_skip)
print("Kết quả:", _result)


Sau khi strip:
''

Chạy validate_smtlib_json:
    ⚠️  No JSON structure found in output
Kết quả: None


In [ ]:
# DEBUG 3: đếm token thinking vs json
print(f"Tổng tokens output: {len(_new)}")
print(f"Có </think> không: {'</think>' in _raw_skip}")

# Tìm vị trí </think> trong token ids
_think_close = "</think>"
_idx = _raw_skip.find(_think_close)
print(f"Vị trí </think> trong string: {_idx}")  # -1 = không có

# Thử với max_new_tokens lớn hơn nhiều
print("\nThử generate với max_new_tokens=6000...")
with model.disable_adapter():
    with torch.no_grad():
        _out2 = model.generate(
            _ids,
            max_new_tokens=6000,
            max_length=None,
            temperature=0.0,
            do_sample=False,
            repetition_penalty=1.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

_new2 = _out2[0][_ids.shape[1]:]
_raw2 = tokenizer.decode(_new2, skip_special_tokens=True)
print(f"Tổng tokens với 6000: {len(_new2)}")
print(f"Có </think> không: {'</think>' in _raw2}")
_idx2 = _raw2.find("</think>")
print(f"Vị trí </think>: {_idx2}")
if _idx2 > 0:
    print(f"Số ký tự thinking: {_idx2}")
    print(f"Phần sau </think>: {repr(_raw2[_idx2:_idx2+200])}")

Tổng tokens output: 3072
Có </think> không: False
Vị trí </think> trong string: -1

Thử generate với max_new_tokens=6000...


In [33]:
# ============================================================
#  EVALUATION v5 — MCQ Multi-Goal + Z3-proved Unknown + Stateful Feedback
# ============================================================
print("📊 Evaluating toàn bộ 411 indices với pipeline v5 (kiến trúc tối thượng)...")
print("=" * 70)

correct           = 0
total_eval        = 0
unknown_correct   = 0
unknown_total     = 0
results_log_v2    = []
iteration_counter = Counter()
z3_status_counter = Counter()

for idx_i, entry in enumerate(raw_data):
    for q_i, (q, expected_ans, _) in enumerate(
        zip(entry["questions"], entry["answers"], entry["explanation"])
    ):
        result = predict_with_z3_loop(
            model          = model,
            tokenizer      = tokenizer,
            fol_list       = entry["premises-FOL"],
            nl_list        = entry["premises-NL"],
            question       = q,
            max_new_tokens = 3072,  # BUG#1 FIX: 300 quá nhỏ → JSON bị cắt giữa chừng
            temperature    = 0.0,
        )

        pred       = result["answer"]
        is_correct = (pred.strip().lower() == expected_ans.strip().lower())

        total_eval += 1
        if is_correct:
            correct += 1
        if expected_ans.strip().lower() == "unknown":
            unknown_total += 1
            if is_correct:
                unknown_correct += 1

        iteration_counter[result["iterations"]] += 1
        z3_status_counter[result["z3_status"]]  += 1

        results_log_v2.append({
            "idx":        idx_i,
            "q_i":        q_i,
            "expected":   expected_ans,
            "pred":       pred,
            "correct":    is_correct,
            "iterations": result["iterations"],
            "z3_status":  result["z3_status"],
        })

        if (idx_i + 1) % 50 == 0 and q_i == 0:
            running_acc = correct / total_eval * 100
            print(f"  [{idx_i+1:3d}/411] Acc: {running_acc:.1f}%  "
                  f"| Iters: {dict(iteration_counter)}  "
                  f"| Z3: {dict(z3_status_counter)}")

print("\n" + "=" * 70)
print("✅ FINAL EVALUATION RESULTS — v5 Pipeline")
print("=" * 70)
overall_acc = correct / total_eval * 100 if total_eval > 0 else 0
print(f"Overall Accuracy  : {correct}/{total_eval} = {overall_acc:.1f}%")

if unknown_total > 0:
    unk_acc = unknown_correct / unknown_total * 100
    print(f"Unknown Accuracy  : {unknown_correct}/{unknown_total} = {unk_acc:.1f}%")

known_total   = total_eval - unknown_total
known_correct = correct - unknown_correct
if known_total > 0:
    known_acc = known_correct / known_total * 100
    print(f"Known Accuracy    : {known_correct}/{known_total} = {known_acc:.1f}%")

print("\n--- Chiến lược 4: Iteration Distribution ---")
for itr, cnt in sorted(iteration_counter.items()):
    pct = cnt / total_eval * 100
    bar = "█" * int(pct / 3)
    print(f"  Iterations={itr}: {cnt:4d} ({pct:5.1f}%) {bar}")

print("\n--- Chiến lược 2: Z3 Status Distribution ---")
for status, cnt in sorted(z3_status_counter.items()):
    pct = cnt / total_eval * 100
    print(f"  {status:12s}: {cnt:4d} ({pct:5.1f}%)")

fallback_count = sum(1 for r in results_log_v2
                     if r['pred'] == FALLBACK_ANSWER)
print(f"\nFallback activated: {fallback_count}/{total_eval} ({fallback_count/total_eval*100:.1f}%)")

if overall_acc >= 60:
    print(f"\n🎯 TARGET MET: {overall_acc:.1f}% ≥ 60% ✅")
else:
    print(f"\n⚠️  Target not met: {overall_acc:.1f}% < 60%")

📊 Evaluating toàn bộ 411 indices với pipeline v5 (kiến trúc tối thượng)...
  ↵ Iter 1/3...     ⚠️  No JSON structure found in output
Z3=error → Retrying...
  ↵ Iter 2/3...     ⚠️  JSON parse failed: Expecting ',' delimiter: line 44 column 12 (char 845)
Z3=error → Retrying...
  ↵ Iter 3/3...     ⚠️  JSON parse failed: Expecting ',' delimiter: line 59 column 12 (char 1178)
Z3=error   ↵ Iter 1/3...     ⚠️  No JSON structure found in output
Z3=error → Retrying...
  ↵ Iter 2/3...     ⚠️  JSON parse failed: Expecting ',' delimiter: line 47 column 12 (char 919)
Z3=error → Retrying...
  ↵ Iter 3/3... 

KeyboardInterrupt: 

In [ ]:
# Phân tích lỗi để hiểu pattern
wrong_samples = [r for r in results_log_v2 if not r["correct"]]
print(f"\n🔍 Error Analysis ({len(wrong_samples)} wrong predictions)")
print("=" * 50)

# Confusion matrix
from collections import defaultdict
confusion = defaultdict(Counter)
for r in results_log_v2:
    confusion[r["expected"]][r["pred"]] += 1

print("\nConfusion (expected → predicted):")
for expected_label in sorted(confusion.keys()):
    row = confusion[expected_label]
    total_row = sum(row.values())
    print(f"  {expected_label:8s} (n={total_row:3d}): ", end="")
    for pred_label, count in sorted(row.items()):
        print(f"{pred_label}={count}", end="  ")
    print()

# Mẫu sai đầu tiên
print(f"\n--- 3 mẫu sai đầu tiên ---")
for r in wrong_samples[:3]:
    entry = raw_data[r["idx"]]
    q     = entry["questions"][r["q_i"]]
    print(f"  idx={r['idx']}, q={r['q_i']}")
    print(f"  Q: {q[:80]}...")
    print(f"  Expected: {r['expected']} | Predicted: {r['pred']}")
    print()

In [ ]:
# Optional: Push lên HuggingFace Hub
PUSH_TO_HUB = False
HF_REPO_ID  = "Barakuga/qwen3-8b-logic-lora"

if PUSH_TO_HUB:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = os.environ.get("HF_TOKEN", "")

    if HF_TOKEN:
        from huggingface_hub import login
        login(token=HF_TOKEN)
        model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
        tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
        print(f"✅ Pushed to: https://huggingface.co/{HF_REPO_ID}")
    else:
        print("⚠️  HF_TOKEN not found. Set via Kaggle Secrets.")
else:
    print("ℹ️  PUSH_TO_HUB=False. Đặt True để push lên HuggingFace Hub.")